# 03 — Arazi Örtüsü Sınıflandırma Modelleri

Hocanın istediği **Accuracy, Precision, Recall, F1 ve ROC-AUC** metrikleri bu notebook'tan çıkar.

**Görev:** Bir Landsat pikselinin 11 özelliğine (6 bant + 5 indeks; NDMI, NDBI'nin tam tersi olduğu için çıkarıldı) bakarak o pikselin 6 sınıftan hangisi olduğunu tahmin etmek: Orman, Tarım, Yapılaşma, Çayır/Çalı, Çıplak, Su/Sulak alan. Buna **denetimli sınıflandırma** denir. "Denetimli" denmesinin nedeni, modelin doğru cevapları (WorldCover etiketleri) görerek öğrenmesidir.

**Akış:**
1. Veriyi eğitim (%70), doğrulama (%15) ve test (%15) olarak böl.
2. Her model için eğitim verisinde hiperparametre araması yap (RandomizedSearchCV, 3 katlı CV).
3. Modelleri **doğrulama** kümesinde karşılaştırıp en iyisini seç.
4. Tüm modelleri bir kez **test** kümesinde değerlendir ve raporla.
5. En iyi model için karışıklık matrisi, ROC eğrileri, özellik önemi ve SHAP.

Earth Engine kullanılmaz. Colab'ın normal CPU'su yeterli, GPU gerekmez.

## Kurulum: kod GitHub'dan, sonuçlar Drive'a
- **Kod** (`config.py`, `src/`) GitHub reposundan Colab'ın geçici diskine (`/content/yapay-zeka-proje`) çekilir. Her çalıştırmada en güncel sürüm alınır (`git pull`).
- **Veri ve sonuçlar** (`data/`, `results/`) Google Drive'da `Drive'ım/yapay-zeka-proje/` altına yazılır. Colab kapansa da kaybolmaz.
- Repo herkese açık olduğu için **token gerekmez**. Colab'ın **Gizli anahtarlar (Secrets)** bölümünde `GITHUB_TOKEN` tanımlıysa yine kullanılır (repo gizli yapılırsa gerekir); token ekrana yazdırılmaz.

In [ ]:
!pip install -q shap

In [ ]:
import os, sys, subprocess, importlib

GITHUB_REPO = "selinzeykaya/yapay-zeka-proje"
GITHUB_BRANCH = "main"   # kendi dalınızdaki kodu çalıştırmak için dal adını yazın (ör. "amine/geoclimate-uyum")
CODE_DIR = "/content/yapay-zeka-proje"                 # kod (GitHub'dan)
WORK_DIR = "/content/drive/MyDrive/yapay-zeka-proje"   # veri + sonuçlar (Drive)

def _git(args, token):
    r = subprocess.run(["git"] + args, capture_output=True, text=True)
    if r.returncode != 0:
        mesaj = r.stderr or r.stdout
        raise RuntimeError("git hatası: " + (mesaj.replace(token, "***") if token else mesaj))

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    # Repo herkese açık: token gerekmez. Colab Secrets'ta GITHUB_TOKEN varsa kullanılır (repo gizli yapılırsa gerekir).
    try:
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        token = ""
    adres = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"
    if not os.path.exists(os.path.join(CODE_DIR, ".git")):
        _git(["clone", "-q", adres, CODE_DIR], token)
    # Seçilen dalın en güncel hâline geç (Colab kopyasındaki yerel değişiklikler atılır)
    _git(["-C", CODE_DIR, "fetch", "-q", "origin", GITHUB_BRANCH], token)
    _git(["-C", CODE_DIR, "checkout", "-q", "-B", GITHUB_BRANCH, f"origin/{GITHUB_BRANCH}"], token)
    os.makedirs(WORK_DIR, exist_ok=True)
else:   # yerel bilgisayarda: notebooks/ klasörünün bir üstü
    CODE_DIR = WORK_DIR = os.path.abspath("..")

sys.path.insert(0, CODE_DIR)
os.chdir(WORK_DIR)
import config
importlib.reload(config)
for d in [config.DATA_RAW, config.DATA_PROCESSED, config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
commit = subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--format=%h %cd", "--date=short"],
                        capture_output=True, text=True).stdout.strip()
print("Kod     :", CODE_DIR, "| dal:", GITHUB_BRANCH if IN_COLAB else "yerel", "| sürüm:", commit)
print("Sonuçlar:", WORK_DIR)

In [ ]:
import json, time, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report, roc_curve, auc)
from sklearn.preprocessing import label_binarize
from sklearn.inspection import permutation_importance

from src import ml_utils as mu
importlib.reload(mu)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

FORCE_RETRAIN = False
T, F, M = config.TABLES_DIR, config.FIGURES_DIR, config.MODELS_DIR
os.makedirs(M, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"{F}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

df = pd.read_csv(f"{config.DATA_PROCESSED}/samples_2021.csv")
FEATS = config.MODEL_FEATURES   # NDMI = −NDBI olduğu için çıkarıldı (bkz. 02)
LABELS = sorted(config.CLASSES)
CLASS_NAMES = [config.CLASSES[k] for k in LABELS]
MODELS = config.MODELS
print(df.shape, "| modeller:", MODELS)

## 1. Model seçimi ve gerekçeleri
Uzaktan algılama literatüründe sık kullanılan, birbirinden farklı çalışma mantığına sahip dört model seçtik. Böylece karşılaştırma yalnızca "hangi ayar daha iyi" değil, "hangi yaklaşım bu veri için daha uygun" sorusuna da cevap veriyor.

In [ ]:
rationale = pd.DataFrame([
    ("RandomForest", "Topluluk (bagging) — çok sayıda karar ağacının oylaması",
     "Uzaktan algılamada en yaygın temel model; aykırı değere ve korelasyonlu özelliklere dayanıklı; "
     "ölçekleme gerektirmez; özellik önemi verir",
     "Çok büyük ağaçlarda bellek; olasılık tahminleri bazen kaba"),
    ("XGBoost", "Topluluk (boosting) — ağaçlar sırayla, öncekinin hatasını düzelterek eklenir",
     "Tablo verisinde genelde en güçlü yöntemlerden; düzenlileştirme ile aşırı öğrenmeyi kontrol eder",
     "Daha çok hiperparametre; yanlış ayarla aşırı öğrenebilir"),
    ("SVM", "Çekirdek (RBF) yöntemi — sınıflar arasında en geniş boşluklu sınırı arar",
     "Küçük–orta veri setlerinde güçlü; klasik uzaktan algılama literatüründe çok kullanılmış, karşılaştırma için önemli",
     "Ölçekleme şart; büyük veride yavaş; olasılık için ek hesap gerekir"),
    ("MLP", "Yapay sinir ağı — gizli katmanlarla doğrusal olmayan ilişkileri öğrenir",
     "Derin öğrenmeye giriş niteliğinde; esnek karar sınırları",
     "Ölçekleme şart; hiperparametreye duyarlı; yorumlanması zor"),
], columns=["Model", "Yaklaşım", "Neden seçildi (artılar)", "Eksiler"])
rationale.to_csv(f"{T}/model_gerekceleri.csv", index=False, encoding="utf-8-sig")
pd.set_option("display.max_colwidth", None)
rationale

## 2. Veriyi bölme: eğitim / doğrulama / test
- **Eğitim (%70):** Model bu veriyle öğrenir, hiperparametre araması da burada yapılır.
- **Doğrulama (%15):** Modeller arasında seçim yapmak için kullanılır.
- **Test (%15):** En sona saklanır ve yalnızca bir kez, nihai raporlama için kullanılır. Testi seçim için kullansaydık test sonucu iyimser olurdu, çünkü dolaylı olarak teste göre ayar yapmış olurduk.

**Katmanlı (stratified) bölme:** Bölmeyi "şehir + sınıf" birleşimine göre katmanlı yapıyoruz. Böylece her kümede her şehrin her sınıfı aynı oranda bulunur.

**Veri sızıntısı önlemleri:**
1. Noktalar arasında ≥ 300 m mesafe var (Faz 1), yani yan yana pikseller farklı kümelere düşmüyor.
2. Ölçekleme (StandardScaler) bir **Pipeline** içinde. Ölçekleyicinin ortalaması ve standart sapması yalnızca eğitim verisinden hesaplanıyor, test verisi hiç "görülmüyor".
3. Test kümesi model seçiminde kullanılmıyor.

Bölme sonucu (`sample_id → train/val/test`) dosyaya kaydediliyor. Faz 4'te Earth Engine modeli **aynı test noktalarında** değerlendirilecek.

In [ ]:
strat = df["city"] + "_" + df["class"].astype(str)
idx_train, idx_tmp = train_test_split(df.index, test_size=config.VAL_SIZE + config.TEST_SIZE,
                                      stratify=strat, random_state=config.RANDOM_STATE)
rel_test = config.TEST_SIZE / (config.VAL_SIZE + config.TEST_SIZE)
idx_val, idx_test = train_test_split(idx_tmp, test_size=rel_test, stratify=strat[idx_tmp],
                                     random_state=config.RANDOM_STATE)
df["split"] = "train"
df.loc[idx_val, "split"] = "val"
df.loc[idx_test, "split"] = "test"
df[["sample_id", "city", "class", "split"]].to_csv(f"{config.DATA_PROCESSED}/split_assignments.csv", index=False)

X_train, y_train = df.loc[idx_train, FEATS], df.loc[idx_train, "class"]
X_val, y_val = df.loc[idx_val, FEATS], df.loc[idx_val, "class"]
X_test, y_test = df.loc[idx_test, FEATS], df.loc[idx_test, "class"]

split_tab = pd.crosstab(df["class_name"], df["split"]).reindex(CLASS_NAMES)[["train", "val", "test"]]
split_tab.loc["Toplam"] = split_tab.sum()
split_tab.to_csv(f"{T}/veri_bolme.csv", encoding="utf-8-sig")
split_tab

## 3. Hiperparametre araması
**Hiperparametre**, modelin veriden öğrenmediği, bizim önceden seçtiğimiz ayarlardır. Örnekler: ağaç sayısı, ağaç derinliği, SVM'deki C değeri.

**RandomizedSearchCV:** Tanımladığımız aralıklardan rastgele **10 kombinasyon** dener. Her kombinasyonu **3 katlı çapraz doğrulama** ile puanlar: eğitim verisi 3 parçaya bölünür, model 2 parçada eğitilip 1 parçada test edilir ve bu 3 kez tekrarlanır. Sonuçların ortalaması o kombinasyonun puanıdır. Tek bir bölmenin şansına bağlı kalmamak için CV kullanırız. Tüm kombinasyonları denemek (GridSearch) çok daha uzun sürerdi.

**Puan ölçütü: macro-F1.** Her sınıfın F1'i eşit ağırlıkla ortalanır, böylece az örnekli sınıflardaki başarısızlık gizlenmez.

Eğitilen modeller `results/models/` altına kaydedilir. Notebook tekrar çalıştırılırsa yeniden eğitilmez, kayıttan yüklenir. Baştan eğitmek için `FORCE_RETRAIN = True` yapın.

In [ ]:
cv = StratifiedKFold(n_splits=config.SEARCH_CV_FOLDS, shuffle=True, random_state=config.RANDOM_STATE)
models, search_rows = {}, []

for name in MODELS:
    path = f"{M}/{name}.joblib"
    if os.path.exists(path) and not FORCE_RETRAIN:
        saved = joblib.load(path)
        models[name] = saved["model"]
        search_rows.append(saved["search_row"])
        print(f"{name}: kayıttan yüklendi")
        continue
    t0 = time.time()
    search = RandomizedSearchCV(mu.make_pipeline(name), mu.param_space(name),
                                n_iter=config.SEARCH_N_ITER, cv=cv, scoring=config.SEARCH_SCORING,
                                n_jobs=-1, random_state=config.RANDOM_STATE, refit=True)
    search.fit(X_train, y_train)
    elapsed = time.time() - t0
    best = search.best_index_
    row = {
        "model": name,
        "arama_araligi": json.dumps(config.PARAM_SPACES[name], default=str),
        "en_iyi_parametreler": json.dumps(mu.strip_prefix(search.best_params_), default=str),
        "cv_macro_f1_ort": search.cv_results_["mean_test_score"][best],
        "cv_macro_f1_std": search.cv_results_["std_test_score"][best],
        "denenen_kombinasyon": len(search.cv_results_["params"]),
        "sure_sn": round(elapsed, 1),
    }
    cvres = pd.DataFrame(search.cv_results_)
    cvres = cvres[[c for c in cvres if c.startswith("param_")] + ["mean_test_score", "std_test_score", "rank_test_score", "mean_fit_time"]]
    cvres.sort_values("rank_test_score").to_csv(f"{T}/hp_cv_sonuclari_{name}.csv", index=False, encoding="utf-8-sig")
    models[name] = search.best_estimator_
    search_rows.append(row)
    joblib.dump({"model": search.best_estimator_, "search_row": row}, path, compress=3)
    print(f"{name}: CV macro-F1 = {row['cv_macro_f1_ort']:.4f} ± {row['cv_macro_f1_std']:.4f} "
          f"| {elapsed:.0f} sn | {row['en_iyi_parametreler']}")

hp_table = pd.DataFrame(search_rows).round(4)
hp_table.to_csv(f"{T}/hiperparametre_arama.csv", index=False, encoding="utf-8-sig")
hp_table

## 4. Değerlendirme metrikleri
Önce kavramlar. Bir sınıf için, örneğin "Su":
- **TP** (doğru pozitif): Gerçekte su, model de su dedi.
- **FP** (yanlış pozitif, yanlış alarm): Gerçekte su değil, model su dedi.
- **FN** (yanlış negatif, kaçırma): Gerçekte su, model başka bir şey dedi.

| Metrik | Formül | Sorusu |
|---|---|---|
| **Accuracy** | doğru tahmin / tüm tahmin | Toplamda yüzde kaçını doğru bildik? |
| **Precision** | TP / (TP + FP) | "Su" dediklerimizin yüzde kaçı gerçekten su? |
| **Recall** | TP / (TP + FN) | Gerçek suların yüzde kaçını bulabildik? |
| **F1** | 2 · P · R / (P + R) | Precision ve Recall'un dengeli ortalaması (harmonik ortalama) |
| **ROC-AUC** | ROC eğrisinin altındaki alan | Model olasılıklarıyla sınıfları ne kadar iyi sıralıyor? 0.5 = rastgele, 1 = mükemmel |
| **Kappa** | (gözlenen uyum − şans uyumu) / (1 − şans uyumu) | Şans eseri doğru tahminler çıkarıldığında kalan başarı |
| **IoU** (Jaccard) | TP / (TP + FP + FN) | Tahmin edilen ve gerçek alanlar ne kadar örtüşüyor? U-Net gibi segmentasyon modellerinin standart metriği; ileride aynı ölçüyle karşılaştırma sağlar |

**Macro mu, weighted mı?** Çok sınıflı problemde her sınıf için ayrı Precision/Recall/F1 hesaplanır. **Macro** bunların düz ortalamasıdır: her sınıf eşit önemli. **Weighted** ise örnek sayısına göre ağırlıklı ortalamadır. Verimiz dengeli olduğu için ikisi birbirine yakın çıkmalı. Ana metrik olarak macro'yu kullanıyoruz.

**ROC-AUC (one-vs-rest, macro):** ROC-AUC aslında iki sınıflı problemler için tanımlıdır. 6 sınıf için her sınıfı "bu sınıf mı, değil mi" diye ayrı ayrı ele alıyoruz (one-vs-rest), 6 AUC hesaplıyoruz ve düz ortalamasını alıyoruz (macro). Accuracy yalnızca nihai tahmine bakar. AUC ise modelin verdiği olasılıkların kalitesini ölçer.

Aşağıda önce **doğrulama** kümesi (model seçimi için), sonra **test** kümesi (raporlama için) değerlendiriliyor.

In [ ]:
def evaluate(X, y, split_name):
    rows, preds, probas = [], {}, {}
    for name, model in models.items():
        p = model.predict(X)
        pr = model.predict_proba(X)
        preds[name], probas[name] = p, pr
        rows.append({"model": name, **mu.compute_metrics(y, p, pr, LABELS)})
    res = pd.DataFrame(rows).set_index("model").round(4)
    return res, preds, probas

val_res, _, _ = evaluate(X_val, y_val, "val")
val_res.to_csv(f"{T}/model_karsilastirma_dogrulama.csv", encoding="utf-8-sig")
BEST = val_res["f1_macro"].idxmax()
print("Doğrulama kümesine göre en iyi model (macro-F1):", BEST)
val_res

In [ ]:
test_res, test_preds, test_probas = evaluate(X_test, y_test, "test")
test_res.to_csv(f"{T}/model_karsilastirma.csv", encoding="utf-8-sig")

# Faz 4'te Earth Engine modeliyle aynı noktalarda karşılaştırma için
pred_df = df.loc[idx_test, ["sample_id", "city", "class"]].copy()
for name in MODELS:
    pred_df[f"pred_{name}"] = test_preds[name]
pred_df.to_csv(f"{config.DATA_PROCESSED}/test_predictions.csv", index=False)

print("TEST kümesi sonuçları (raporlanacak tablo):")
test_res

### En iyi modeli kaydet
En iyi model **doğrulama** sonucuna göre seçildi. Kaydedilen dosyayla Faz 4 (2014/2024 haritaları) ve Faz 6 (genelleme) aynı modeli ve aynı ayarları kullanır.

In [ ]:
best_params = json.loads(hp_table.set_index("model").loc[BEST, "en_iyi_parametreler"])
joblib.dump(models[BEST], f"{M}/best_model.joblib", compress=3)
meta = {
    "best_model": BEST, "selected_by": "validation f1_macro",
    "params": best_params, "features": FEATS, "classes": config.CLASSES,
    "test_metrics": test_res.loc[BEST].to_dict(),
    "all_params": {m: json.loads(v) for m, v in hp_table.set_index("model")["en_iyi_parametreler"].items()},
}
with open(f"{M}/best_model.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2, default=str)
print(json.dumps(meta, ensure_ascii=False, indent=2, default=str))

## 5. Model karşılaştırma grafiği
Hocanın istediği beş metrik, dört model için test kümesinde.

In [ ]:
show = {"accuracy": "Accuracy", "precision_macro": "Precision (macro)", "recall_macro": "Recall (macro)",
        "f1_macro": "F1 (macro)", "roc_auc_ovr_macro": "ROC-AUC (OvR macro)"}
plot_df = test_res[list(show)].rename(columns=show)
ax = plot_df.T.plot(kind="bar", figsize=(12, 5), width=0.8, colormap="Set2")
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=7, rotation=90, padding=2)
lo = max(0, plot_df.values.min() - 0.1)
ax.set_ylim(lo, 1.02); ax.set_ylabel("Skor"); ax.set_title("Model karşılaştırması (test kümesi)")
ax.tick_params(axis="x", rotation=0); ax.legend(title="Model", loc="lower right")
savefig("model_karsilastirma")

**Neye bakmalıyız:** Modeller arasındaki fark büyük mü, yoksa hepsi birbirine yakın mı? (Yakınsa "veri iyi hazırlanmış, model seçimi ikincil" yorumu yapılabilir.) Ağaç modelleri (RF, XGBoost) SVM ve MLP'den farklı mı? ROC-AUC'nin diğer metriklerden yüksek çıkması normaldir, çünkü sıralama başarısı sınıflandırma başarısından daha kolay bir hedeftir.

**Y ekseni uyarısı:** Farklar görünsün diye eksen 0'dan başlamıyor. Raporda bunu belirtin.

## 6. Karışıklık matrisleri
Satırlar **gerçek sınıfı**, sütunlar **modelin tahminini** gösterir. Köşegen doğru tahminlerdir. Köşegen dışındaki her hücre "şu sınıfı şununla karıştırdı" anlamına gelir. Satır yüzdesi kullanıyoruz: her satır %100'e tamamlanır ve köşegen o sınıfın Recall değeridir.

In [ ]:
def plot_cm(ax, y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    cm_pct = 100 * cm / cm.sum(axis=1, keepdims=True)
    annot = np.array([[f"{p:.0f}%\n({n})" for p, n in zip(rp, rn)] for rp, rn in zip(cm_pct, cm)])
    sns.heatmap(cm_pct, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=100, cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax, annot_kws={"size": 8})
    ax.set_xlabel("Tahmin"); ax.set_ylabel("Gerçek"); ax.set_title(title)
    ax.tick_params(axis="x", rotation=35); ax.tick_params(axis="y", rotation=0)
    return cm

cms = {}
for name in MODELS:
    fig, ax = plt.subplots(figsize=(7, 6))
    cms[name] = plot_cm(ax, y_test, test_preds[name], f"{name} — karışıklık matrisi (test)")
    savefig(f"karisiklik_matrisi_{name}")
    pd.DataFrame(cms[name], index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        f"{T}/karisiklik_matrisi_{name}.csv", encoding="utf-8-sig")

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, name in zip(axes.ravel(), MODELS):
    plot_cm(ax, y_test, test_preds[name], name)
savefig("karisiklik_matrisleri_tumu")

**Neye bakmalıyız:** En çok hangi sınıf çiftleri karışıyor? Uzaktan algılamada tipik karışmalar şunlardır: Yapılaşma ↔ Çıplak (ikisi de bitkisiz ve parlak), Tarım ↔ Çayır/Çalı (hasat edilmiş tarla ile kuru çayır benzer görünür), Orman ↔ Çayır/Çalı (çalılık). Faz 2'deki kutu grafiklerinde üst üste binen sınıflarla karşılaştırın.

## 7. Sınıf bazında rapor
Her sınıf için Precision, Recall, F1 ve destek (test kümesindeki örnek sayısı).

In [ ]:
f1_by_class = {}
for name in MODELS:
    rep = classification_report(y_test, test_preds[name], labels=LABELS, target_names=CLASS_NAMES,
                                output_dict=True, zero_division=0)
    rep_df = pd.DataFrame(rep).T.round(4)
    rep_df.to_csv(f"{T}/sinif_raporu_{name}.csv", encoding="utf-8-sig")
    f1_by_class[name] = rep_df.loc[CLASS_NAMES, "f1-score"]
    if name == BEST:
        print(f"En iyi model ({BEST}) sınıf raporu:")
        display(rep_df)

f1_tab = pd.DataFrame(f1_by_class)
f1_tab.to_csv(f"{T}/sinif_bazinda_f1.csv", encoding="utf-8-sig")
plt.figure(figsize=(8, 5))
sns.heatmap(f1_tab, annot=True, fmt=".3f", cmap="YlGn", vmin=max(0, f1_tab.values.min() - 0.05), vmax=1)
plt.title("Sınıf bazında F1 (test)"); plt.ylabel("")
savefig("sinif_bazinda_f1")

### 7a. Sınıf bazında IoU
**IoU (Intersection over Union, Jaccard)** = TP / (TP + FP + FN). F1'e benzer ama daha "sert" bir ölçüdür: aynı tahmin için IoU her zaman F1'den küçük ya da ona eşittir (IoU = F1 / (2 − F1)). Segmentasyon (U-Net) çalışmalarında standart metrik olduğu için, ileride piksel bazlı modellerle aynı ölçüyle karşılaştırma yapabilmek amacıyla ekliyoruz.

In [ ]:
from sklearn.metrics import jaccard_score
iou_tab = pd.DataFrame({name: jaccard_score(y_test, test_preds[name], labels=LABELS, average=None, zero_division=0)
                        for name in MODELS}, index=CLASS_NAMES)
iou_tab.loc["Ortalama (macro)"] = iou_tab.mean()
iou_tab = iou_tab.round(4)
iou_tab.to_csv(f"{T}/sinif_bazinda_iou.csv", encoding="utf-8-sig")
plt.figure(figsize=(8, 5))
sns.heatmap(iou_tab, annot=True, fmt=".3f", cmap="YlGn", vmin=max(0, iou_tab.values.min() - 0.05), vmax=1)
plt.title("Sınıf bazında IoU (test)"); plt.ylabel("")
savefig("sinif_bazinda_iou")
iou_tab

## 7b. Su/Sulak alan alt tipleri: otsu sulak alan neye benzetiliyor?
Faz 2'de gördük: WorldCover'ın **otsu sulak alan (90)** noktaları yaz kompozitinde sudan çok bitki gibi görünüyor (NDVI yüksek, MNDWI negatif). Burada test kümesindeki Su/Sulak noktalarını alt tiplerine ayırıp her modelin bunları **hangi sınıfa** tahmin ettiğine bakıyoruz.

- `recall`: Alt tipin doğru olarak Su/Sulak bulunma oranı.
- Diğer sütunlar: Yüzde kaçının Orman, Tarım, Çayır/Çalı vb. olarak tahmin edildiği.

In [ ]:
test_wet = df.loc[idx_test].query("`class` == 5")
rows = []
for name in MODELS:
    pred = pd.Series(test_preds[name], index=idx_test).loc[test_wet.index]
    for code, grp in test_wet.groupby("wc_code"):
        dist = pred.loc[grp.index].map(config.CLASSES).value_counts(normalize=True) * 100
        rows.append({"model": name, "alt_tip": f"{code} {config.WORLDCOVER_NAMES[code]}", "n_test": len(grp),
                     "recall": (pred.loc[grp.index] == 5).mean(), **dist.round(1).to_dict()})
wet_tab = pd.DataFrame(rows).fillna(0)
wet_tab = wet_tab[["model", "alt_tip", "n_test", "recall"] + [c for c in CLASS_NAMES if c in wet_tab]]
wet_tab.round(3).to_csv(f"{T}/su_sulak_alt_tip_tahminleri.csv", index=False, encoding="utf-8-sig")
display(wet_tab.round(3))

b = wet_tab[wet_tab["model"] == BEST].set_index("alt_tip")[[c for c in CLASS_NAMES if c in wet_tab]]
ax = b.plot(kind="barh", stacked=True, figsize=(10, 2.8), colormap="tab10", width=0.7)
ax.set_xlabel("Tahmin dağılımı (%)"); ax.set_ylabel("")
ax.set_title(f"Su/Sulak alan alt tipleri — {BEST} tahminleri (test)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
savefig("su_sulak_alt_tip_tahminleri")

**Neye bakmalıyız ve raporda nasıl yorumlanır:**
- Kalıcı suyun (80) recall değeri yüksek, otsu sulak alanınki (90) belirgin şekilde düşükse, Su/Sulak sınıfındaki hataların kaynağı büyük ölçüde sulak alanlardır. Bu alanlar Çayır/Çalı ya da Tarım olarak tahmin ediliyorsa model "yanlış" değil, **yaz görüntüsünde gördüğünü** sınıflandırıyor demektir. Sulak alan, spektral değil **hidrolojik** bir tanımdır (yılın bir kısmında su basması).
- Bu sonuç, Faz 4'te AS3 (göl/sulak alan küçülmesi) için şunu söyler: Su yüzeyi değişimi için MNDWI tabanlı **açık su** alanı güvenilirdir. Sulak alan değişimi ise tek bir yaz kompoziti ile güvenilir ölçülemez. Çözüm için çok mevsimli kompozit ya da radar (Sentinel-1) verisi gerekir; bunu gelecek çalışmaya yazın.
- Not: Test kümesindeki otsu sulak alan nokta sayısı (`n_test`) küçükse yüzdeler oynak olur. Raporda n'yi de verin.

## 8. ROC eğrileri (çok sınıflı, one-vs-rest)
**ROC eğrisi**, karar eşiği 1'den 0'a indirilirken **yanlış pozitif oranı** (x) ile **doğru pozitif oranı**nın (y) nasıl değiştiğini çizer. Eğri sol üst köşeye ne kadar yakınsa model o kadar iyidir. Köşegen (kesikli çizgi) rastgele tahmini gösterir. Her sınıf için ayrı bir eğri çiziliyor, ayrıca macro ortalama eğrisi ekleniyor.

In [ ]:
y_bin = label_binarize(y_test, classes=LABELS)
roc_rows = []

def plot_roc(ax, name):
    proba = test_probas[name]
    grid_fpr = np.linspace(0, 1, 500)
    tprs = []
    for i, cname in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], proba[:, i])
        a = auc(fpr, tpr)
        roc_rows.append({"model": name, "sinif": cname, "auc": a})
        tprs.append(np.interp(grid_fpr, fpr, tpr))
        ax.plot(fpr, tpr, lw=1.2, label=f"{cname} (AUC={a:.3f})")
    mean_tpr = np.mean(tprs, axis=0)
    ax.plot(grid_fpr, mean_tpr, "k-", lw=2.5, label=f"Macro ort. (AUC={auc(grid_fpr, mean_tpr):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_xlabel("Yanlış pozitif oranı"); ax.set_ylabel("Doğru pozitif oranı")
    ax.set_title(f"{name} — ROC (test)"); ax.legend(fontsize=7, loc="lower right")

for name in MODELS:
    fig, ax = plt.subplots(figsize=(6.5, 6))
    plot_roc(ax, name)
    savefig(f"roc_{name}")

roc_tab = pd.DataFrame(roc_rows).drop_duplicates().pivot(index="sinif", columns="model", values="auc").reindex(CLASS_NAMES)
roc_tab.round(4).to_csv(f"{T}/roc_auc_sinif_bazinda.csv", encoding="utf-8-sig")

fig, axes = plt.subplots(2, 2, figsize=(13, 12))
for ax, name in zip(axes.ravel(), MODELS):
    plot_roc(ax, name)
savefig("roc_tumu")
roc_tab.round(4)

**Not:** Grafikteki macro eğrisinin AUC'si, ortalama eğri üzerinden hesaplandığı için tablodaki `roc_auc_ovr_macro` değerinden (6 AUC'nin ortalaması) üçüncü basamakta farklı olabilir. Raporda tablodaki değeri kullanın.

## 9. Özellik önemi
Hangi bant/indeks sınıflandırmada en çok işe yarıyor? İki yöntem kullanıyoruz:

- **Permütasyon önemi (tüm modeller için):** Test kümesinde bir özelliğin değerleri rastgele karıştırılır ve macro-F1'in ne kadar düştüğüne bakılır. Çok düşüyorsa model o özelliğe güveniyordur. Modelden bağımsız bir yöntemdir.
- **RF safsızlık önemi:** Random Forest'ın ağaçları kurarken her özelliğin ne kadar "ayrım" sağladığının toplamı. Hızlıdır ama korelasyonlu özelliklerde yanıltıcı olabilir.

Hatırlatma (Faz 2): Korelasyonlu özellikler (ör. NDVI–SAVI) önemi aralarında **paylaşır**. Permütasyonda biri karıştırılınca model diğerini kullanmaya devam eder, bu yüzden ikisinin de önemi düşük görünebilir.

In [ ]:
perm_rows = {}
for name in MODELS:
    r = permutation_importance(models[name], X_test, y_test, scoring="f1_macro", n_repeats=10,
                               random_state=config.RANDOM_STATE, n_jobs=-1)
    perm_rows[name] = pd.Series(r.importances_mean, index=FEATS)
perm = pd.DataFrame(perm_rows)
rf_imp = pd.Series(models["RandomForest"].named_steps["clf"].feature_importances_, index=FEATS, name="RF_safsizlik")
imp = perm.add_prefix("perm_").join(rf_imp).round(4).sort_values(f"perm_{BEST}", ascending=False)
imp.to_csv(f"{T}/ozellik_onemi.csv", encoding="utf-8-sig")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
perm.loc[imp.index].plot(kind="barh", ax=axes[0], width=0.8, colormap="Set2")
axes[0].invert_yaxis(); axes[0].set_xlabel("macro-F1 düşüşü"); axes[0].set_title("Permütasyon önemi (test)")
rf_imp.sort_values().plot(kind="barh", ax=axes[1], color="forestgreen")
axes[1].set_title("Random Forest safsızlık önemi"); axes[1].set_xlabel("Önem")
savefig("ozellik_onemi")
imp

## 10. SHAP analizi
**SHAP (SHapley Additive exPlanations)** oyun teorisinden gelen bir yöntemdir. Her tahmin için her özelliğin o tahmine "ne kadar katkı yaptığını" hesaplar. Ortalama |SHAP| değeri büyük olan özellik, model için genel olarak önemlidir. Permütasyon önemine göre artısı şudur: SHAP **sınıf bazında** da bakabilmemizi sağlar. Örneğin "Su sınıfını tanırken en çok MNDWI'ye bakıyor" gibi.

SHAP'ın hızlı ve kesin sürümü (TreeExplainer) yalnızca ağaç modellerinde çalışır. En iyi model SVM veya MLP çıkarsa SHAP'ı en iyi **ağaç** modeli için hesaplarız ve bunu açıkça belirtiriz. Hesap hızlı olsun diye 500 test noktası kullanılıyor.

In [ ]:
import shap
tree_models = [m for m in ["RandomForest", "XGBoost"] if m in MODELS]
SHAP_MODEL = BEST if BEST in tree_models else val_res.loc[tree_models, "f1_macro"].idxmax()
print("SHAP hesaplanan model:", SHAP_MODEL, "" if SHAP_MODEL == BEST else f"(en iyi model {BEST} ağaç tabanlı olmadığı için)")

X_shap = X_test.sample(min(500, len(X_test)), random_state=config.RANDOM_STATE)
explainer = shap.TreeExplainer(models[SHAP_MODEL].named_steps["clf"])
sv = explainer.shap_values(X_shap)
sv = np.stack(sv, axis=-1) if isinstance(sv, list) else np.asarray(sv)   # → (n, özellik, sınıf)
if sv.ndim == 3 and sv.shape[1] != len(FEATS):
    sv = np.transpose(sv, (0, 2, 1))
mean_abs = pd.DataFrame(np.abs(sv).mean(axis=0), index=FEATS, columns=CLASS_NAMES)
mean_abs["Genel"] = mean_abs.mean(axis=1)
mean_abs = mean_abs.sort_values("Genel", ascending=False).round(5)
mean_abs.to_csv(f"{T}/shap_ortalama_mutlak_{SHAP_MODEL}.csv", encoding="utf-8-sig")

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={"width_ratios": [1, 1.6]})
mean_abs["Genel"].sort_values().plot(kind="barh", ax=axes[0], color="slateblue")
axes[0].set_title(f"SHAP — genel önem ({SHAP_MODEL})"); axes[0].set_xlabel("ortalama |SHAP|")
sns.heatmap(mean_abs[CLASS_NAMES], annot=True, fmt=".3f", cmap="Purples", ax=axes[1], annot_kws={"size": 7})
axes[1].set_title("SHAP — sınıf bazında önem"); axes[1].set_ylabel("")
savefig(f"shap_{SHAP_MODEL}")
mean_abs

**Neye bakmalıyız:** Her sınıf için en önemli özellik mantıklı mı? Beklentiler şöyle: Su ↔ MNDWI, Yapılaşma ↔ NDBI/BSI/SWIR bantları, Orman ↔ NDVI/NDMI/NIR. Beklenmedik bir özellik öne çıkıyorsa (ör. tek bir görünür bant) not edin. Bu durum şehirler arası farkları (ör. iklim kaynaklı parlaklık) ezberleme işareti olabilir ve Faz 6'da test edilecek.

## 10b. Mekânsal blok doğrulama: rastgele bölme başarıyı şişiriyor mu?
Bölüm 2'deki bölme **rastgele** (katmanlı). Noktalar arasında ≥ 300 m var, ama aynı mahalledeki, aynı tarladaki ya da aynı ormandaki noktalar hâlâ birbirine çok benziyor. Rastgele bölmede test noktasının "komşusu" eğitimde kalabilir; bu **mekânsal veri sızıntısıdır** ve başarıyı olduğundan yüksek gösterebilir (Roberts ve ark., 2017 — L21; Ploton ve ark., 2020 — L23).

**Test:** Aynı modeller (Bölüm 3'te seçilen hiperparametrelerle) iki şekilde 5 katlı çapraz doğrulamayla değerlendiriliyor:
- **Rastgele:** şehir + sınıf katmanlı, karıştırılmış 5 katman.
- **Mekânsal blok:** harita 10 km × 10 km bloklara bölünüyor; **aynı bloktaki noktalar hep aynı katmanda** (GroupKFold). Test noktalarının komşuları eğitimde olmuyor.

İki sonuç arasındaki fark, rastgele bölmenin ne kadar iyimser olduğunu gösterir. **Ana tablo (Bölüm 4) değişmiyor:** Faz 4 ve Faz 6 o bölmeye ve o modele bağlı. Raporda ikisini yan yana verin; gerçekçi başarı tahmini olarak mekânsal blok sonucunu kullanın.

SVM'de olasılık hesabı kapatıldı (yalnızca tahmin gerekiyor, ROC-AUC hesaplanmıyor), böylece süre kısalıyor. Toplam 40 eğitim; Colab CPU'da yaklaşık 10–20 dakika. Sonuç kaydedilir, tekrar çalıştırmada yeniden hesaplanmaz (`FORCE_RETRAIN = True` ile baştan).

In [ ]:
from pyproj import Transformer
from sklearn.model_selection import GroupKFold

blk_csv = f"{T}/mekansal_blok_dogrulama.csv"
to_m = Transformer.from_crs("EPSG:4326", config.GRID_CRS, always_xy=True)
xm, ym = to_m.transform(df["lon"].values, df["lat"].values)
B = config.SPATIAL_BLOCK_M
bx = pd.Series((xm // B).astype(int), index=df.index).astype(str)
by = pd.Series((ym // B).astype(int), index=df.index).astype(str)
blocks = df["city"] + "_" + bx + "_" + by
print(f"Blok boyu: {B / 1000:g} km | blok sayısı: {blocks.nunique()} | blok başına ortalama nokta: {len(df) / blocks.nunique():.1f}")

X_all, y_all = df[FEATS], df["class"]
SCHEMES = {
    "Rastgele": list(StratifiedKFold(n_splits=5, shuffle=True, random_state=config.RANDOM_STATE).split(X_all, strat)),
    "Mekânsal blok (10 km)": list(GroupKFold(n_splits=5).split(X_all, y_all, groups=blocks)),
}
params_of = {m: json.loads(v) for m, v in hp_table.set_index("model")["en_iyi_parametreler"].items()}

if os.path.exists(blk_csv) and not FORCE_RETRAIN:
    blk = pd.read_csv(blk_csv)
    print("kayıttan yüklendi")
else:
    rows = []
    for scheme, folds in SCHEMES.items():
        for name in MODELS:
            t0 = time.time()
            for k, (tr, te) in enumerate(folds):
                pipe = mu.make_pipeline(name, params_of[name])
                if name == "SVM":
                    pipe.set_params(clf__probability=False)
                pipe.fit(X_all.iloc[tr], y_all.iloc[tr])
                p = pipe.predict(X_all.iloc[te])
                m = mu.compute_metrics(y_all.iloc[te], p, None, LABELS)
                rows.append({"bolme": scheme, "model": name, "katman": k, "n_test": len(te),
                             **{c: m[c] for c in ["accuracy", "f1_macro", "iou_macro", "kappa"]}})
            print(f"{scheme:22s} {name:12s} {time.time() - t0:5.0f} sn")
    blk = pd.DataFrame(rows).round(4)
    blk.to_csv(blk_csv, index=False, encoding="utf-8-sig")

ozet = blk.groupby(["model", "bolme"])[["accuracy", "f1_macro", "iou_macro", "kappa"]].agg(["mean", "std"]).round(4)
ozet.columns = [f"{a}_{b}" for a, b in ozet.columns]
ozet = ozet.reset_index()
ozet.to_csv(f"{T}/mekansal_blok_ozet.csv", index=False, encoding="utf-8-sig")

piv = ozet.pivot(index="model", columns="bolme", values="f1_macro_mean").loc[MODELS]
piv["fark (rastgele − blok)"] = piv["Rastgele"] - piv["Mekânsal blok (10 km)"]
ax = piv[["Rastgele", "Mekânsal blok (10 km)"]].plot(kind="bar", figsize=(9, 4.5), color=["#9ecae1", "#08519c"], width=0.7)
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=8)
ax.set_ylim(max(0, piv.iloc[:, :2].values.min() - 0.1), 1); ax.set_ylabel("macro-F1 (5 katman ortalaması)")
ax.set_title("Rastgele bölme vs mekânsal blok doğrulama"); ax.tick_params(axis="x", rotation=0)
savefig("mekansal_blok_dogrulama")
piv.round(4)

**Nasıl yorumlanır:**
- Fark küçükse (≈ 0.01–0.02), 300 m mesafe filtresi sızıntıyı büyük ölçüde önlemiş demektir; Bölüm 4'teki sonuçlar güvenilirdir.
- Fark büyükse (≥ 0.05), rastgele bölme başarıyı şişiriyor demektir. Raporda gerçekçi başarı olarak mekânsal blok değerini verin ve bunu sınırlılıklara ekleyin.
- Mekânsal blok sonucu, Faz 6'daki "başka şehir" sonucuyla "aynı şehir" sonucu arasında bir yerde olmalı: blok doğrulamada test noktasının şehri eğitimde var ama yakın komşuları yok.
- Katmanlar arası standart sapmaya (`mekansal_blok_ozet.csv`) da bakın: blok doğrulamada katmanlar daha farklı olduğu için std daha büyük çıkabilir.

## 11. Özet

In [ ]:
print("Test kümesi — hocanın istediği metrikler:")
display(test_res[["accuracy", "precision_macro", "recall_macro", "f1_macro", "roc_auc_ovr_macro", "kappa", "iou_macro"]])
print(f"\nEn iyi model (doğrulama macro-F1'e göre): {BEST}")
print(f"Test: Accuracy={test_res.loc[BEST, 'accuracy']:.4f}, macro-F1={test_res.loc[BEST, 'f1_macro']:.4f}, "
      f"ROC-AUC={test_res.loc[BEST, 'roc_auc_ovr_macro']:.4f}, Kappa={test_res.loc[BEST, 'kappa']:.4f}, "
      f"IoU={test_res.loc[BEST, 'iou_macro']:.4f}")
gap = (val_res["f1_macro"] - test_res["f1_macro"]).round(4).rename("dogrulama_eksi_test_f1")
print("\nDoğrulama − test macro-F1 farkı (küçükse sonuçlar kararlı):")
display(gap.to_frame())

## Faz 3 bitti mi?
- [ ] `results/tables/model_karsilastirma.csv`: 4 model × (Accuracy, Precision, Recall, F1, ROC-AUC, Kappa)
- [ ] `results/tables/hiperparametre_arama.csv`: arama aralıkları ve seçilen değerler
- [ ] Her model için `karisiklik_matrisi_<model>.png` ve `roc_<model>.png`
- [ ] `results/models/best_model.joblib` ve `best_model.json`
- [ ] Özellik önemi ve SHAP grafikleri
- [ ] `sinif_bazinda_iou.csv` ve `mekansal_blok_ozet.csv` (rastgele vs mekânsal blok doğrulama)

## Bu fazda ne yaptık, neden yaptık
- Dört farklı yaklaşımı (bagging, boosting, çekirdek yöntemi, sinir ağı) aynı veri ve aynı bölmeyle karşılaştırdık. Karşılaştırmanın adil olması için hepsi aynı CV ve aynı arama bütçesiyle (10 deneme) ayarlandı.
- Veriyi eğitim, doğrulama ve test olarak üçe böldük. Model seçimini doğrulama kümesiyle yaptık. Test kümesine yalnızca son raporlamada dokunduk, böylece test sonucu iyimser olmadı.
- Veri sızıntısını üç yolla önledik: noktalar arası 300 m mesafe, Pipeline içinde yalnızca eğitimle fit edilen ölçekleyici, seçimde test kümesini kullanmamak.
- Beş zorunlu metriği macro ortalamayla raporladık: her sınıf eşit önemli, nadir sınıflardaki hata gizlenmiyor. Kappa'yı da ekledik, çünkü literatürde arazi örtüsü haritaları için yaygın.
- Karışıklık matrisi, sınıf bazında F1, permütasyon önemi ve SHAP ile modelin **neyi neden** karıştırdığını ve hangi özelliklere dayandığını gösterdik.